In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

from IPython.core.display import display, HTML
display(HTML("<style>.output_scroll{height: auto !important;}</style>"))

In [3]:
df=pd.read_excel('Ti6Al4V-NiTi单道沉积试样实验测量熔池形貌数据汇总.xlsx',sheet_name='Sheet1')
df

,试样标号,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,熔池宽度（μm）,熔池深度（μm）,沉积宽度（μm）,沉积高度（μm）,左润湿角（°）,右润湿角（°）
0,A-1,0.0,600,200,1.000000,90.000000,2803.604,836.036,2075.6760,1762.162,131.315,99.836
1,A-2,0.0,700,200,1.000000,105.000000,3189.189,908.108,2428.8290,1751.351,109.671,91.650
2,A-3,0.0,800,200,1.000000,120.000000,3654.054,1001.802,2900.9010,1888.288,104.036,87.592
3,A-4,0.0,900,200,1.000000,135.000000,4075.676,1095.495,3189.1890,1780.180,92.068,98.973
4,A-5,0.0,600,250,0.600000,72.000000,2661.111,807.477,1824.0740,1659.259,138.094,134.676
5,A-6,0.0,700,250,0.600000,84.000000,3124.074,914.815,2301.8520,1648.148,132.679,112.818
6,A-7,0.0,800,250,0.600000,96.000000,3549.550,936.937,2724.3240,1517.117,99.851,85.486
7,A-8,0.0,900,250,0.600000,108.000000,3960.360,1045.045,3099.0990,1513.514,70.434,87.241
8,A-9,0.0,600,300,0.333333,60.000000,2631.776,798.131,2016.8220,1343.925,85.851,128.480
9,A-10,0.0,700,300,0.333333,70.000000,3018.692,829.907,2366.3550,1338.486,96.607,81.174


## Predict Molten Pool Width

In [4]:
df_width = df.drop(df.columns[[0, 7, 8, 9, 10, 11]], axis=1)
df_width

,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,熔池宽度（μm）
0,0.0,600,200,1.000000,90.000000,2803.604
1,0.0,700,200,1.000000,105.000000,3189.189
2,0.0,800,200,1.000000,120.000000,3654.054
3,0.0,900,200,1.000000,135.000000,4075.676
4,0.0,600,250,0.600000,72.000000,2661.111
5,0.0,700,250,0.600000,84.000000,3124.074
6,0.0,800,250,0.600000,96.000000,3549.550
7,0.0,900,250,0.600000,108.000000,3960.360
8,0.0,600,300,0.333333,60.000000,2631.776
9,0.0,700,300,0.333333,70.000000,3018.692


In [5]:
df_width.describe()

,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,熔池宽度（μm）
count,120.000000,120.000000,120.00000,120.000000,120.000000,120.000000
mean,0.500000,608.333333,300.00000,0.415238,64.570238,2814.882050
std,0.342997,167.574558,71.00716,0.356285,24.505039,633.983643
min,0.000000,350.000000,200.00000,0.000000,26.250000,1672.222000
25%,0.200000,487.500000,250.00000,0.142857,45.000000,2293.332500
50%,0.500000,600.000000,300.00000,0.333333,60.000000,2800.867500
75%,0.800000,725.000000,350.00000,0.600000,77.857143,3319.342750
max,1.000000,900.000000,400.00000,1.000000,135.000000,4198.198000


In [6]:
X = df_width.iloc[:, :-1]
y = df_width.iloc[:, -1]

In [7]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from tabpfn import TabPFNRegressor 

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

regressor = TabPFNRegressor()  
regressor.fit(X_train, y_train)

y_pred = regressor.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'TabPFN模型在测试集上的均方根误差: {rmse:.4f}')
print('TabPFN模型在测试集上的R²得分:', r2)

TabPFN模型在测试集上的均方根误差: 60.4359
TabPFN模型在测试集上的R²得分: 0.9894767984207724


In [9]:
from tabpfn_extensions.post_hoc_ensembles.sklearn_interface import AutoTabPFNRegressor

reg = AutoTabPFNRegressor(max_time=300, device="cuda") 
reg.fit(X_train, y_train)

predictions = reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f'TabPFN_Post Hoc Ensembling模型在测试集上的均方根误差: {rmse:.4f}')
print('TabPFN_Post Hoc Ensembling模型在测试集上的R²得分:', r2)

INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Order of selections: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(4), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(15)]
INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Val loss over iterations: [np.float64(1760.2568372879714), np.float64(1760.2568372879714), np.float64(1760.2568372879714), np.float64(1760.2568372879714), np.float64(1760.2568372879714), np.float64(1760.2568372879714), np.float64(1760.2568372879714), np.float64(1760.2568372879714), np.float64(1760.2568372879714), np.float64(1759.7607936731954), np.float64(1759.2978661459529), np.float64(1758.98970774012), np.float64(1758.784069231196), np.float64(1758.6480161231657), np.float64(1758.5601257545995), np.float64(175

TabPFN_Post Hoc Ensembling模型在测试集上的均方根误差: 51.9656
TabPFN_Post Hoc Ensembling模型在测试集上的R²得分: 0.9922198046906245


## Predict Molten Pool Depth

In [10]:
df_depth = df.drop(df.columns[[0, 6, 8, 9, 10, 11]], axis=1)
df_depth

,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,熔池深度（μm）
0,0.0,600,200,1.000000,90.000000,836.036
1,0.0,700,200,1.000000,105.000000,908.108
2,0.0,800,200,1.000000,120.000000,1001.802
3,0.0,900,200,1.000000,135.000000,1095.495
4,0.0,600,250,0.600000,72.000000,807.477
5,0.0,700,250,0.600000,84.000000,914.815
6,0.0,800,250,0.600000,96.000000,936.937
7,0.0,900,250,0.600000,108.000000,1045.045
8,0.0,600,300,0.333333,60.000000,798.131
9,0.0,700,300,0.333333,70.000000,829.907


In [11]:
df_depth.describe()

,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,熔池深度（μm）
count,120.000000,120.000000,120.00000,120.000000,120.000000,120.000000
mean,0.500000,608.333333,300.00000,0.415238,64.570238,812.593842
std,0.342997,167.574558,71.00716,0.356285,24.505039,191.163796
min,0.000000,350.000000,200.00000,0.000000,26.250000,418.519000
25%,0.200000,487.500000,250.00000,0.142857,45.000000,664.838500
50%,0.500000,600.000000,300.00000,0.333333,60.000000,806.337000
75%,0.800000,725.000000,350.00000,0.600000,77.857143,954.296500
max,1.000000,900.000000,400.00000,1.000000,135.000000,1257.658000


In [12]:
X = df_depth.iloc[:, :-1]
y = df_depth.iloc[:, -1]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

regressor = TabPFNRegressor()  
regressor.fit(X_train, y_train)

y_pred = regressor.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'TabPFN模型在测试集上的均方根误差: {rmse:.4f}')
print('TabPFN模型在测试集上的R²得分:', r2)

TabPFN模型在测试集上的均方根误差: 50.9300
TabPFN模型在测试集上的R²得分: 0.8948190105106684


In [14]:
reg = AutoTabPFNRegressor(max_time=300, device="cuda") 
reg.fit(X_train, y_train)

predictions = reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f'TabPFN_Post Hoc Ensembling模型在测试集上的均方根误差: {rmse:.4f}')
print('TabPFN_Post Hoc Ensembling模型在测试集上的R²得分:', r2)

INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Order of selections: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(7), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Val loss over iterations: [np.float64(2231.5124397667787), np.float64(2231.5124397667787), np.float64(2231.5124397667787), np.float64(2231.5124397667787), np.float64(2231.5124397667787), np.float64(2231.5124397667787), np.float64(2231.5124397667787), np.float64(2231.5124397667787), np.float64(2231.5124397667787), np.float64(2231.5124397667787), np.float64(2231.5124397667787), np.float64(2231.5124397667787), np.float64(2231.5124397667787), np.float64(2231.299477189367), np.float64(2231.0458770787395), np.float64(22

TabPFN_Post Hoc Ensembling模型在测试集上的均方根误差: 52.7243
TabPFN_Post Hoc Ensembling模型在测试集上的R²得分: 0.8872771496946492


## Predict Deposited Width

In [15]:
df_deposited_width = df.drop(df.columns[[0, 6, 7, 9, 10, 11]], axis=1)
df_deposited_width

,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,沉积宽度（μm）
0,0.0,600,200,1.000000,90.000000,2075.6760
1,0.0,700,200,1.000000,105.000000,2428.8290
2,0.0,800,200,1.000000,120.000000,2900.9010
3,0.0,900,200,1.000000,135.000000,3189.1890
4,0.0,600,250,0.600000,72.000000,1824.0740
5,0.0,700,250,0.600000,84.000000,2301.8520
6,0.0,800,250,0.600000,96.000000,2724.3240
7,0.0,900,250,0.600000,108.000000,3099.0990
8,0.0,600,300,0.333333,60.000000,2016.8220
9,0.0,700,300,0.333333,70.000000,2366.3550


In [16]:
df_deposited_width.describe()

,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,沉积宽度（μm）
count,120.000000,120.000000,120.00000,120.000000,120.000000,120.000000
mean,0.500000,608.333333,300.00000,0.415238,64.570238,2361.604826
std,0.342997,167.574558,71.00716,0.356285,24.505039,543.593936
min,0.000000,350.000000,200.00000,0.000000,26.250000,1388.580000
25%,0.200000,487.500000,250.00000,0.142857,45.000000,1918.941000
50%,0.500000,600.000000,300.00000,0.333333,60.000000,2335.692000
75%,0.800000,725.000000,350.00000,0.600000,77.857143,2817.754750
max,1.000000,900.000000,400.00000,1.000000,135.000000,3447.273000


In [17]:
X = df_deposited_width.iloc[:, :-1]
y = df_deposited_width.iloc[:, -1]

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

regressor = TabPFNRegressor()  
regressor.fit(X_train, y_train)

y_pred = regressor.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'TabPFN模型在测试集上的均方根误差: {rmse:.4f}')
print('TabPFN模型在测试集上的R²得分:', r2)

TabPFN模型在测试集上的均方根误差: 81.3803
TabPFN模型在测试集上的R²得分: 0.9736208011456411


In [19]:
reg = AutoTabPFNRegressor(max_time=300, device="cuda") 
reg.fit(X_train, y_train)

predictions = reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f'TabPFN_Post Hoc Ensembling模型在测试集上的均方根误差: {rmse:.4f}')
print('TabPFN_Post Hoc Ensembling模型在测试集上的R²得分:', r2)

INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Order of selections: [np.int64(0), np.int64(4), np.int64(0), np.int64(18), np.int64(0), np.int64(4), np.int64(0), np.int64(4), np.int64(0), np.int64(18), np.int64(0), np.int64(4), np.int64(0), np.int64(4), np.int64(0), np.int64(0), np.int64(18), np.int64(0), np.int64(4), np.int64(0), np.int64(4), np.int64(0), np.int64(18), np.int64(0), np.int64(4)]
INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Val loss over iterations: [np.float64(6248.270013414831), np.float64(5243.940754766223), np.float64(5243.940754766223), np.float64(5212.474199984879), np.float64(5212.474199984879), np.float64(5199.877567973351), np.float64(5199.877567973351), np.float64(5199.877567973351), np.float64(5199.877567973351), np.float64(5199.877567973351), np.float64(5196.055467212743), np.float64(5196.055467212743), np.float64(5194.839219655718), np.float64(5194.839219655718), np.float64(5194.839219655718), np.float64(5194.8392196

TabPFN_Post Hoc Ensembling模型在测试集上的均方根误差: 87.8026
TabPFN_Post Hoc Ensembling模型在测试集上的R²得分: 0.9692929222377004


## Predict Deposited Height

In [20]:
df_deposited_height = df.drop(df.columns[[0, 6, 7, 8, 10, 11]], axis=1)
df_deposited_height

,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,沉积高度（μm）
0,0.0,600,200,1.000000,90.000000,1762.162
1,0.0,700,200,1.000000,105.000000,1751.351
2,0.0,800,200,1.000000,120.000000,1888.288
3,0.0,900,200,1.000000,135.000000,1780.180
4,0.0,600,250,0.600000,72.000000,1659.259
5,0.0,700,250,0.600000,84.000000,1648.148
6,0.0,800,250,0.600000,96.000000,1517.117
7,0.0,900,250,0.600000,108.000000,1513.514
8,0.0,600,300,0.333333,60.000000,1343.925
9,0.0,700,300,0.333333,70.000000,1338.486


In [21]:
df_deposited_height.describe()

,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,沉积高度（μm）
count,120.000000,120.000000,120.00000,120.000000,120.000000,120.000000
mean,0.500000,608.333333,300.00000,0.415238,64.570238,1133.842025
std,0.342997,167.574558,71.00716,0.356285,24.505039,361.631371
min,0.000000,350.000000,200.00000,0.000000,26.250000,427.778000
25%,0.200000,487.500000,250.00000,0.142857,45.000000,921.403250
50%,0.500000,600.000000,300.00000,0.333333,60.000000,1103.704000
75%,0.800000,725.000000,350.00000,0.600000,77.857143,1377.846250
max,1.000000,900.000000,400.00000,1.000000,135.000000,1896.364000


In [22]:
X = df_deposited_height.iloc[:, :-1]
y = df_deposited_height.iloc[:, -1]

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

regressor = TabPFNRegressor()  
regressor.fit(X_train, y_train)

y_pred = regressor.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'TabPFN模型在测试集上的均方根误差: {rmse:.4f}')
print('TabPFN模型在测试集上的R²得分:', r2)

TabPFN模型在测试集上的均方根误差: 107.9830
TabPFN模型在测试集上的R²得分: 0.9294142200391884


In [24]:
reg = AutoTabPFNRegressor(max_time=300, device="cuda") 
reg.fit(X_train, y_train)

predictions = reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f'TabPFN_Post Hoc Ensembling模型在测试集上的均方根误差: {rmse:.4f}')
print('TabPFN_Post Hoc Ensembling模型在测试集上的R²得分:', r2)

INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Order of selections: [np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1)]
INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Val loss over iterations: [np.float64(4303.89680736023), np.float64(4282.398034231542), np.float64(4268.750621627684), np.float64(4268.750621627684), np.float64(4268.750621627684), np.float64(4268.750621627684), np.float64(4268.674279920884), np.float64(4268.674279920884), np.float64(4268.674279920884), np.float64(4268.518779544518), np.float64(4268.518779544518), np.float64(4268.518779544518), np.float64(4268.498386876207), np.float64(4268.498386876207), np.float64(4268.498386876207), np.float64(4268.498386876207

TabPFN_Post Hoc Ensembling模型在测试集上的均方根误差: 103.8703
TabPFN_Post Hoc Ensembling模型在测试集上的R²得分: 0.9346884833439262


## Predict Left Wetting Angle

In [25]:
df_left = df.drop(df.columns[[0, 11]], axis=1)
df_left

,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,熔池宽度（μm）,熔池深度（μm）,沉积宽度（μm）,沉积高度（μm）,左润湿角（°）
0,0.0,600,200,1.000000,90.000000,2803.604,836.036,2075.6760,1762.162,131.315
1,0.0,700,200,1.000000,105.000000,3189.189,908.108,2428.8290,1751.351,109.671
2,0.0,800,200,1.000000,120.000000,3654.054,1001.802,2900.9010,1888.288,104.036
3,0.0,900,200,1.000000,135.000000,4075.676,1095.495,3189.1890,1780.180,92.068
4,0.0,600,250,0.600000,72.000000,2661.111,807.477,1824.0740,1659.259,138.094
5,0.0,700,250,0.600000,84.000000,3124.074,914.815,2301.8520,1648.148,132.679
6,0.0,800,250,0.600000,96.000000,3549.550,936.937,2724.3240,1517.117,99.851
7,0.0,900,250,0.600000,108.000000,3960.360,1045.045,3099.0990,1513.514,70.434
8,0.0,600,300,0.333333,60.000000,2631.776,798.131,2016.8220,1343.925,85.851
9,0.0,700,300,0.333333,70.000000,3018.692,829.907,2366.3550,1338.486,96.607


In [26]:
df_left.describe()

,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,熔池宽度（μm）,熔池深度（μm）,沉积宽度（μm）,沉积高度（μm）,左润湿角（°）
count,120.000000,120.000000,120.00000,120.000000,120.000000,120.000000,120.000000,120.000000,120.000000,120.000000
mean,0.500000,608.333333,300.00000,0.415238,64.570238,2814.882050,812.593842,2361.604826,1133.842025,82.600308
std,0.342997,167.574558,71.00716,0.356285,24.505039,633.983643,191.163796,543.593936,361.631371,23.734517
min,0.000000,350.000000,200.00000,0.000000,26.250000,1672.222000,418.519000,1388.580000,427.778000,29.964000
25%,0.200000,487.500000,250.00000,0.142857,45.000000,2293.332500,664.838500,1918.941000,921.403250,68.168500
50%,0.500000,600.000000,300.00000,0.333333,60.000000,2800.867500,806.337000,2335.692000,1103.704000,82.958500
75%,0.800000,725.000000,350.00000,0.600000,77.857143,3319.342750,954.296500,2817.754750,1377.846250,96.937250
max,1.000000,900.000000,400.00000,1.000000,135.000000,4198.198000,1257.658000,3447.273000,1896.364000,138.094000


In [27]:
X = df_left.iloc[:, :-1]
y = df_left.iloc[:, -1]

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

regressor = TabPFNRegressor()  
regressor.fit(X_train, y_train)

y_pred = regressor.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'TabPFN模型在测试集上的均方根误差: {rmse:.4f}')
print('TabPFN模型在测试集上的R²得分:', r2)

TabPFN模型在测试集上的均方根误差: 9.5983
TabPFN模型在测试集上的R²得分: 0.7778543057813112


In [29]:
reg = AutoTabPFNRegressor(max_time=300, device="cuda") 
reg.fit(X_train, y_train)

predictions = reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f'TabPFN_Post Hoc Ensembling模型在测试集上的均方根误差: {rmse:.4f}')
print('TabPFN_Post Hoc Ensembling模型在测试集上的R²得分:', r2)

INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Order of selections: [np.int64(0), np.int64(1), np.int64(2), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(2), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(2), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(2)]
INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Val loss over iterations: [np.float64(76.24864911774179), np.float64(71.72972808511199), np.float64(71.72972808511199), np.float64(71.72972808511199), np.float64(71.5811181554637), np.float64(71.5811181554637), np.float64(71.55081857349064), np.float64(71.55081857349064), np.float64(71.55081857349064), np.float64(71.55081857349064), np.float64(71.55081857349064), np.float64(71.55081857349064), np.float64(71.55081857349064), np.float64(71.55081857349064), np.float64(71.55081857349064), np.float64(71.55081857349064)

TabPFN_Post Hoc Ensembling模型在测试集上的均方根误差: 9.5435
TabPFN_Post Hoc Ensembling模型在测试集上的R²得分: 0.7803842474128695


## Predict Right Wetting Angle

In [30]:
df_right = df.drop(df.columns[[0, 10]], axis=1)
df_right

,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,熔池宽度（μm）,熔池深度（μm）,沉积宽度（μm）,沉积高度（μm）,右润湿角（°）
0,0.0,600,200,1.000000,90.000000,2803.604,836.036,2075.6760,1762.162,99.836
1,0.0,700,200,1.000000,105.000000,3189.189,908.108,2428.8290,1751.351,91.650
2,0.0,800,200,1.000000,120.000000,3654.054,1001.802,2900.9010,1888.288,87.592
3,0.0,900,200,1.000000,135.000000,4075.676,1095.495,3189.1890,1780.180,98.973
4,0.0,600,250,0.600000,72.000000,2661.111,807.477,1824.0740,1659.259,134.676
5,0.0,700,250,0.600000,84.000000,3124.074,914.815,2301.8520,1648.148,112.818
6,0.0,800,250,0.600000,96.000000,3549.550,936.937,2724.3240,1517.117,85.486
7,0.0,900,250,0.600000,108.000000,3960.360,1045.045,3099.0990,1513.514,87.241
8,0.0,600,300,0.333333,60.000000,2631.776,798.131,2016.8220,1343.925,128.480
9,0.0,700,300,0.333333,70.000000,3018.692,829.907,2366.3550,1338.486,81.174


In [31]:
df_right.describe()

,NiTi粉末材料质量比例,激光功率（W）,移动速度（mm/min）,归一化线质量,激光能量密度（J/mm2）,熔池宽度（μm）,熔池深度（μm）,沉积宽度（μm）,沉积高度（μm）,右润湿角（°）
count,120.000000,120.000000,120.00000,120.000000,120.000000,120.000000,120.000000,120.000000,120.000000,120.000000
mean,0.500000,608.333333,300.00000,0.415238,64.570238,2814.882050,812.593842,2361.604826,1133.842025,73.919267
std,0.342997,167.574558,71.00716,0.356285,24.505039,633.983643,191.163796,543.593936,361.631371,20.509268
min,0.000000,350.000000,200.00000,0.000000,26.250000,1672.222000,418.519000,1388.580000,427.778000,29.943000
25%,0.200000,487.500000,250.00000,0.142857,45.000000,2293.332500,664.838500,1918.941000,921.403250,60.060750
50%,0.500000,600.000000,300.00000,0.333333,60.000000,2800.867500,806.337000,2335.692000,1103.704000,75.049500
75%,0.800000,725.000000,350.00000,0.600000,77.857143,3319.342750,954.296500,2817.754750,1377.846250,85.610750
max,1.000000,900.000000,400.00000,1.000000,135.000000,4198.198000,1257.658000,3447.273000,1896.364000,134.676000


In [32]:
X = df_right.iloc[:, :-1]
y = df_right.iloc[:, -1]

In [33]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

regressor = TabPFNRegressor()  
regressor.fit(X_train, y_train)

y_pred = regressor.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'TabPFN模型在测试集上的均方根误差: {rmse:.4f}')
print('TabPFN模型在测试集上的R²得分:', r2)

TabPFN模型在测试集上的均方根误差: 7.4306
TabPFN模型在测试集上的R²得分: 0.8215659069220371


In [34]:
reg = AutoTabPFNRegressor(max_time=300, device="cuda") 
reg.fit(X_train, y_train)

predictions = reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f'TabPFN_Post Hoc Ensembling模型在测试集上的均方根误差: {rmse:.4f}')
print('TabPFN_Post Hoc Ensembling模型在测试集上的R²得分:', r2)

INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Order of selections: [np.int64(0), np.int64(1), np.int64(4), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(4), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(4), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(4), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1)]
INFO:tabpfn_extensions.post_hoc_ensembles.greedy_weighted_ensemble:Val loss over iterations: [np.float64(67.44759609805234), np.float64(66.29054188438548), np.float64(66.2750706754481), np.float64(66.19072227706255), np.float64(66.12909759414549), np.float64(66.12909759414549), np.float64(66.12864446333313), np.float64(66.12864446333313), np.float64(66.12864446333313), np.float64(66.12864446333313), np.float64(66.12864446333313), np.float64(66.12430464983595), np.float64(66.12430464983595), np.float64(66.12430464983595), np.float64(66.12430464983595), np.float64(66.12430464983595

TabPFN_Post Hoc Ensembling模型在测试集上的均方根误差: 7.2693
TabPFN_Post Hoc Ensembling模型在测试集上的R²得分: 0.8292291324023033
